## Dataset Summary

In [ ]:
from pathlib import Path
from collections import Counter

DATASET_DIR = Path("dataset")

# ── Dataset summary ───────────────────────────────────────────────────────────
dest_imgs = DATASET_DIR / "images" / "all"
dest_lbls = DATASET_DIR / "labels" / "all"

all_imgs = sorted(dest_imgs.glob("*.jpg"))
print(f"Total frames: {len(all_imgs)}")

# Per-match breakdown
from collections import Counter
slugs = Counter()
for img in all_imgs:
    # Extract slug: everything before _frame_
    parts = img.stem.split("_frame_")
    slug = parts[0] if len(parts) == 2 else "unknown"
    slugs[slug] += 1

print(f"\n{'Match':<15s} {'Frames':>6s}")
print("-" * 25)
for slug, count in sorted(slugs.items()):
    print(f"{slug:<15s} {count:>6d}")

# Label stats
total_persons = 0
total_balls = 0
for lbl in dest_lbls.glob("*.txt"):
    content = lbl.read_text().strip()
    if not content:
        continue
    for line in content.split("\n"):
        if line.startswith("0 "):
            total_persons += 1
        elif line.startswith("1 "):
            total_balls += 1

print(f"\nAnnotations: {total_persons} persons, {total_balls} balls")

# Train/val counts
train_n = len(list((DATASET_DIR / "images" / "train").glob("*.jpg")))
val_n = len(list((DATASET_DIR / "images" / "val").glob("*.jpg")))
print(f"Split: {train_n} train, {val_n} val")

## Visual QA — Browse Labels

Interactive tool to spot-check annotations. Filter by match, scroll through frames,
or jump to a random frame. Green boxes = person, cyan = ball.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

DATASET_DIR = Path("dataset")
IMGS = DATASET_DIR / "images" / "all"
LBLS = DATASET_DIR / "labels" / "all"
CLASSES = {0: "person", 1: "ball"}
COLORS = {0: (0, 255, 0), 1: (0, 255, 255)}

all_imgs = sorted(IMGS.glob("*.jpg"))
by_match = defaultdict(list)
for img in all_imgs:
    parts = img.stem.split("_frame_")
    slug = parts[0] if len(parts) == 2 else "unknown"
    by_match[slug].append(img)

match_slugs = ["all"] + sorted(by_match.keys())

def draw_labels(img_path):
    frame = cv2.imread(str(img_path))
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w = frame.shape[:2]
    lbl_path = LBLS / img_path.with_suffix(".txt").name
    n_person = n_ball = 0
    if lbl_path.exists():
        content = lbl_path.read_text().strip()
        if content:
            for line in content.splitlines():
                parts = line.split()
                cls = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                color = COLORS.get(cls, (255, 255, 255))
                thickness = 3 if cls == 1 else 2
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
                label = CLASSES.get(cls, f"cls{cls}")
                cv2.putText(frame, label, (x1, y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
                if cls == 0: n_person += 1
                elif cls == 1: n_ball += 1
    return frame, n_person, n_ball

match_dd = widgets.Dropdown(options=match_slugs, value="all", description="Match:")
idx_slider = widgets.IntSlider(value=0, min=0, max=len(all_imgs)-1, description="Frame:", layout=widgets.Layout(width="80%"))
rand_btn = widgets.Button(description="Random", button_style="info")
info_label = widgets.HTML()
out = widgets.Output()
current_list = [all_imgs]

def update_match(change):
    slug = match_dd.value
    current_list[0] = all_imgs if slug == "all" else by_match[slug]
    idx_slider.max = max(0, len(current_list[0]) - 1)
    idx_slider.value = 0
    show_frame(None)

def show_frame(change):
    imgs = current_list[0]
    if not imgs:
        return
    idx = min(idx_slider.value, len(imgs) - 1)
    img_path = imgs[idx]
    frame, n_p, n_b = draw_labels(img_path)
    info_label.value = f"<b>{img_path.name}</b> &nbsp; | &nbsp; {n_p} persons, {n_b} balls &nbsp; | &nbsp; {idx+1}/{len(imgs)}"
    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(1, 1, figsize=(16, 9))
        ax.imshow(frame)
        ax.axis("off")
        plt.tight_layout()
        plt.show()

def random_frame(btn):
    idx_slider.value = np.random.randint(0, len(current_list[0]))

match_dd.observe(update_match, names="value")
idx_slider.observe(show_frame, names="value")
rand_btn.on_click(random_frame)

display(widgets.HBox([match_dd, rand_btn]))
display(idx_slider)
display(info_label)
display(out)
show_frame(None)
